# 02 — Silver: typed, cleaned, deduplicatedWhere judgement gets applied. Four things happen:1. Strings become real types; the literal `'null'` becomes a true NULL2. Sentinel dates are resolved3. Duplicate client rows (contract renewals) are collapsed4. **`is_bulk_load` is derived** — the single transformation that changes   every conclusion downstreamRedundant string date columns (`created_at_str`, `closed_at_str`) are droppedhere rather than at ingestion, so Bronze stays a faithful copy of the source.

In [0]:
from pyspark.sql import functions as F, Windowspark.sql("CREATE SCHEMA IF NOT EXISTS silver.bid")bronze_bids = spark.table("bronze.bid.bids")bronze_clients = spark.table("bronze.bid.clients")

## Null handlingThe export writes the four-character string `'null'`, and `'-'` for anabsent closure date. Neither is a NULL to Spark, so both would silentlysurvive every downstream filter.

In [0]:
def denull(df, placeholders=("null", "-", "")):    """Replace placeholder strings with true NULLs across all string columns."""    for c, t in df.dtypes:        if t == "string":            df = df.withColumn(                c, F.when(F.trim(F.col(c)).isin(list(placeholders)), None)                    .otherwise(F.col(c))            )    return dfbids = denull(bronze_bids)clients = denull(bronze_clients)

## Typing`outcome` is deliberately left nullable: NULL means the bid is still open,which is a distinct state from lost and must not collapse into `0`. Roughly28% of the table sits in that state.

In [0]:
bids_typed = (    bids    .withColumn("bid_id", F.col("bid_id").cast("long"))    .withColumn("client_id", F.col("client_id").cast("long"))    .withColumn("created_at", F.to_timestamp("created_at"))    .withColumn("bid_date", F.to_timestamp("bid_date"))    .withColumn("closed_at", F.to_timestamp("closed_at"))    .withColumn("outcome", F.col("outcome").cast("int"))          # NULL = open    .withColumn("is_confirmed_date", F.col("is_confirmed_date").cast("int"))    .withColumn("contract_value_brl", F.col("contract_value_brl").cast("double"))    .drop("created_at_str", "closed_at_str"))

## Deriving `is_bulk_load`A large share of rows were mass-imported during a system migration ratherthan entered as bids happened. They share an identical `created_at` down tothe second, and they behave nothing like organic bids — a far lower win rate,concentrated in specific portfolios.Left unflagged, they poison every segmented metric: the executive who ownedthe migrated portfolio looks like the worst performer in the company purelybecause of how their records were loaded.The threshold of 10 is a judgement call. Two bids registered in the samesecond is plausible; ten is not.

In [0]:
BULK_THRESHOLD = 10batch = Window.partitionBy("created_at")bids_flagged = (    bids_typed    .withColumn("_batch_size", F.count("*").over(batch))    .withColumn("is_bulk_load", (F.col("_batch_size") >= BULK_THRESHOLD).cast("boolean"))    .drop("_batch_size"))

## Loss reason coverage`competitor_name` carries a default value written whenever nobody completedthe post-mortem. Flagging it explicitly stops it being counted as a realcompetitor in any downstream aggregate — which would otherwise produce thefalse headline that one competitor takes the overwhelming majority of losses.

In [0]:
PLACEHOLDER_COMPETITOR = "Competitor 1"bids_clean = (    bids_flagged    .withColumn(        "competitor_is_placeholder",        (F.col("competitor_name") == F.lit(PLACEHOLDER_COMPETITOR)).cast("boolean"),    )    .withColumn("has_loss_reason", F.col("loss_reason").isNotNull())    .withColumn(        "bid_status",        F.when(F.col("outcome") == 1, "won")         .when(F.col("outcome") == 0, "lost")         .otherwise("open"),    ))bids_clean.write.format("delta").mode("overwrite").saveAsTable("silver.bid.bids_clean")

## Clients: sentinel dates and duplicates`2999-12-31` marks an open-ended contract; keeping it as a date would put a977-year contract into any duration calculation. It becomes NULL alongside anexplicit `is_open_ended` flag.Renewals are recorded as a second row for the same `client_id`. Deduplicatingon the most recent contract keeps one row per client, which is what the jointo `bids` requires — an un-deduplicated dimension would fan out the fact tableand inflate every count.

In [0]:
SENTINEL = "2999-12-31"clients_typed = (    clients    .withColumn("client_id", F.col("client_id").cast("long"))    .withColumn("is_open_ended", (F.col("end_date").startswith(SENTINEL)).cast("boolean"))    .withColumn(        "end_date",        F.when(F.col("end_date").startswith(SENTINEL), None)         .otherwise(F.to_date("end_date")),    )    .withColumn("start_date", F.to_date("start_date")))latest = Window.partitionBy("client_id").orderBy(F.col("start_date").desc_nulls_last())clients_clean = (    clients_typed    .withColumn("_rn", F.row_number().over(latest))    .filter(F.col("_rn") == 1)    .drop("_rn"))clients_clean.write.format("delta").mode("overwrite").saveAsTable("silver.bid.clients_clean")

## Referential integrityA bid pointing at a client that does not exist would be dropped silently byan inner join later. Checking here means it surfaces as a number, not as aquietly shrinking row count.

In [0]:
orphans = (    bids_clean.join(clients_clean, "client_id", "left_anti").count())dupes = (    clients_clean.groupBy("client_id").count().filter("count > 1").count())print(f"orphan bids: {orphans}")print(f"duplicate client_ids after dedup: {dupes}")assert dupes == 0, "clients_clean must have one row per client_id"